# Ensemblr: 1D Umbrella Sampling Setup (adaptive ensemble)

This notebook takes an ensemble written by the adaptive sampler (`ensemblr-generate` /
`generate_af2_ensemble_v2.py`) through to 1D umbrella sampling windows along PC1. The MC path
optimisation scores each step between neighbouring windows by backbone (Ramachandran φ/ψ) and
sidechain (Janin χ1/χ2) distance, plus even spacing along PC1. All user-configurable parameters
are in **Section 1 (Setup)**.

The notebook expects the sampler's output directory (`ensemble_dir`):

```
ensemble_dir/
├── structures/        # symlinks to every accepted model (r{round}_[cell_{i}_{j}_]{paired}-{unpaired}_rank_NNN.pdb)
├── plddt_dict.json    # {filename: pLDDT} for everything in structures/
├── rounds/            # raw ColabFold outputs per round (not read here)
└── run_01/            # written by this notebook (out_dir): PCA, matrices, MC paths, umbrella_sampling/
```

Steps that take a while (PCA, structural properties, matrices, MC) save checkpoints in `out_dir`
and reload them on re-runs. Change `run_name` to start a separate analysis of the same ensemble.

| Section | Description |
|---------|-------------|
| 1. Setup | Imports and all configurable parameters |
| 2. Load Ensemble | Read `structures/` + `plddt_dict.json`, filter by pLDDT and atom count |
| 3. Selection & PCA | Generate RMSD/PCA selection; run dimensionality reduction |
| 4. Clustering | Identify representative structures via HDBSCAN |
| 5. Structural Properties | RMSD, SASA, and user-defined distances |
| 6. Visualisation | Ensemble density and property plots |
| 7. Property Matrices | Pairwise Rama, Janin and CV matrices |
| 8. Path Optimisation | Bin structures along PC1 and run MC simulated annealing |
| 9. Path Assessment | Evaluate quality of the optimised path |
| 10. Umbrella Sampling Setup | Embed structures and write PLUMED input files |
| 11. Debugging | Verify PLUMED CV values against MDA projections |

## 1. Setup

All imports and user-configurable parameters live here. Edit the **SETTINGS** block before running the notebook — everything below the settings block runs automatically.

In [ ]:
# ── Suppress warnings / logging BEFORE any imports ───────────────────────────
import warnings
warnings.filterwarnings('ignore')

import os
os.environ["JAX_PLATFORMS"] = "cpu"       # prevent JAX GPU-search spam
os.environ["OMP_NUM_THREADS"] = "1"       # placeholder; overwritten below

# ── Imports ───────────────────────────────────────────────────────────────────
import glob
import json
import pickle
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import jax

from IPython.display import display
from ensemblr import Ensemble, MCPathFinder, UmbrellaSamplingPrep

# Silence noisy loggers that bypass Python's warnings system
for _log in ('absl', 'jax', 'jax._src.xla_bridge', 'MDAnalysis', 'numexpr'):
    logging.getLogger(_log).setLevel(logging.ERROR)

# ── SETTINGS ──────────────────────────────────────────────────────────────────

# Output directory of the adaptive sampler (the output_dir given to ensemblr-generate)
ensemble_dir        = "/path/to/target/synthetic_ensemble/"
structure_directory = ensemble_dir + "structures/"       # symlinks written by the sampler
plddt_dict_path     = ensemble_dir + "plddt_dict.json"

# Run name — all outputs for this run go to out_dir; change to keep runs separate
run_name = "run_01"
out_dir  = ensemble_dir + run_name + "/"

# pLDDT threshold — structures below this value are discarded (can be raised above the
# threshold used during generation, but not lowered: those models were never kept)
thresh_pLDDT = 85

# Residue selections for RMSD and PCA calculations (MDAnalysis syntax, e.g. "(resid 215-300)")
conserved_residues = ""
ignored_residues   = ""
always_include     = ""
include_loops      = True  # set False to exclude loop residues from the PCA selection

# Optional: named inter-atomic distances to compute and add to the ensemble dataframe
distances_to_evaluate = {}   # e.g. {'IF_gate': ("resid 30 and name CA", "resid 160 and name CA")}

# Optional: external reference structures to compute RMSD against for every ensemble structure.
# Format: {label: absolute_path} — adds a 'rmsd_to_{label}' column to ens.df for each entry.
# Use these to identify which ensemble structures are closest to known end states.
# Set user_ref_resid_offset if the reference structures have different residue numbering than
# the ensemble (e.g. crystal structures offset by a signal peptide).
user_specified_structures = {}   # e.g. {'IF': '/path/to/IF.pdb', 'OF': '/path/to/OF.pdb'}
user_ref_resid_offset     = 0    # residue ID offset applied to user-specified reference structures

# Toggle optional per-structure calculations
calculate_rmsd      = True
calculate_sasa      = False
calculate_distances = False

# CPU cores for parallelised steps
num_processes = 6

# Number of principal components to retain
n_pcs = 3

# Collective variable for MC path optimisation and umbrella sampling
collective_variable   = 'PC1'
collective_variable_2 = None   # optional orthogonal CV (e.g. 'PC2') for the ODF term; None = off

# Step-cost matrix for MC path optimisation: 'rama' (backbone φ/ψ) or 'rmsd'
step_matrix_type = 'rama'

# MC path optimisation settings
mc_n_bins          = 32    # number of umbrella sampling windows
mc_n_runs          = 100   # number of independent MC runs
mc_n_paths_to_save = 5     # number of top-ranked paths to write as PDB files
reproducible_seeds = False
fixed_endstates    = False  # freeze end-state structures during MC
mc_wf_step         = 1.0   # weight for step-cost (Rama/RMSD) term
mc_wf_cv           = 1.0   # weight for even spacing along the CV
mc_wf_odf          = 0.0   # weight for the CV2 term; None → auto (PC2_eig/PC1_eig) when CV2 is a PC
mc_wf_janin        = 1.0   # weight for sidechain (Janin χ1/χ2) smoothness over the PCA selection; 0 = off

# Template PDB for umbrella sampling (protein + membrane + solvent, protein as chain A),
# ideally the last frame of an equilibrated simulation with a working topology.
# Leave blank to stop after the path (Section 10 is skipped)
template_pdb_for_umbrella_sampling = ''
resid_offset = 0  # offset applied to residue IDs when writing output structures

# Default axes for ensemble plots
plot_variable_1 = 'PC1'
plot_variable_2 = 'PC2'

# ── END SETTINGS ──────────────────────────────────────────────────────────────

os.makedirs(out_dir, exist_ok=True)
os.environ["OMP_NUM_THREADS"] = str(num_processes)

pd.set_option('display.max_colwidth', None)

cmap   = plt.colormaps['tab20c']
colors = cmap(np.linspace(0, 1, 10))
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=colors)

# ── Shared formatting helper ──────────────────────────────────────────────────
def _banner(title, width=64):
    bar = '─' * width
    print(f'\n{bar}\n  {title}\n{bar}')

# ── Summary ───────────────────────────────────────────────────────────────────
_banner("1. Setup")
print(f"  Output dir       : {out_dir}")
print(f"  pLDDT threshold  : {thresh_pLDDT}")
print(f"  CPU cores        : {num_processes}")
print(f"  Ensemble dir     : {ensemble_dir}")
print(f"  CV               : {collective_variable}")
print(f"  Step matrix      : {step_matrix_type}  (+ Janin, wf={mc_wf_janin})")
print(f"  JAX devices      : {jax.devices()}")

## 2. Load Ensemble

Reads the structures the adaptive sampler linked into `structures/` and their pLDDT scores from
`plddt_dict.json`, filters out structures below `thresh_pLDDT`, and drops any structures with fewer
than the maximum atom count to ensure topological consistency.

In [ ]:
_banner("2. Load Ensemble")

ens = Ensemble(structure_directory, thresh_pLDDT=thresh_pLDDT)
with open(plddt_dict_path) as f:
    plddt_dict = json.load(f)
ens.load(plddt_dict=plddt_dict)

print(f"\n  {len(ens.df)} structures loaded")
print(f"  pLDDT range : {ens.df['pLDDT'].min():.1f} – {ens.df['pLDDT'].max():.1f}")
#display(ens.df)

## 3. Selection & PCA

Generates an MDAnalysis selection token using DSSP secondary structure information from the highest-pLDDT structure, then runs PCA on the aligned ensemble. PC projections are stored in `ens.df`. Visualisation PDBs for each PC are written to `output_dir`.

In [ ]:
import types

_banner("3. Selection & PCA")

ens.build_selection(
    conserved_residues=conserved_residues,
    excluded_residues=ignored_residues,
    always_include=always_include,
    include_loops=include_loops,
    output_file=out_dir + 'rmsd_selection.pdb',
)
print(f"  Selection : {ens.rmsd_selection}")

_pca_checkpoint = out_dir + 'pca_checkpoint.npz'
pc_cols = [f'PC{i+1}' for i in range(n_pcs)]
_pca_valid = False

if os.path.exists(_pca_checkpoint):
    _npz = np.load(_pca_checkpoint)
    if 'PC1' in _npz and _npz['PC1'].shape[0] == len(ens.df):
        for c in pc_cols:
            ens.df[c] = _npz[c]
        ens.pca = types.SimpleNamespace(
            p_components=_npz['p_components'],
            mean=_npz['mean'],
            cumulated_variance=_npz['cumulated_variance'],
            variance=(_npz['variance'] if 'variance' in _npz
                     else np.diff(np.concatenate([[0], _npz['cumulated_variance']]))),
        )
        ens._ensemble_pdb = out_dir + 'ensemble.pdb'
        cumvar = {i+1: round(float(v)*100, 1) for i, v in enumerate(_npz['cumulated_variance'][:n_pcs])}
        print(f"  Loaded from checkpoint\n  Cumulative variance (%): {cumvar}")
        _pca_valid = True
    else:
        print("  Checkpoint size mismatch — will recompute")

if not _pca_valid:
    ens.run_pca(n_components=n_pcs, output_dir=out_dir)
    np.savez(
        _pca_checkpoint,
        p_components=ens.pca.p_components,
        mean=np.asarray(ens.pca.mean).flatten(),
        cumulated_variance=np.asarray(list(ens.pca.cumulated_variance)),
        variance=np.asarray(list(ens.pca.variance[:n_pcs])),
        **{c: ens.df[c].values for c in pc_cols},
    )
    print(f"  Saved checkpoint: {_pca_checkpoint}")

print(f"\n  PC projections (first 5 rows):")
#display(ens.df[pc_cols].head())

# Auto-set mc_wf_odf from eigenvalue ratio when CV2 is a PC
if mc_wf_odf is None:
    if (collective_variable_2 is not None
            and collective_variable_2.startswith('PC')
            and collective_variable.startswith('PC')):
        _cv1_idx = int(collective_variable[2:]) - 1
        _cv2_idx = int(collective_variable_2[2:]) - 1
        mc_wf_odf = float(ens.pca.variance[_cv2_idx] / ens.pca.variance[_cv1_idx])
        print(f'  Auto wf_odf = {mc_wf_odf:.3f}  '
              f'(variance ratio {collective_variable_2}/{collective_variable})')
    else:
        mc_wf_odf = 0.0


## 4. Clustering

Clusters the ensemble in PC space using HDBSCAN (minimum samples = 10% of ensemble size). Cluster medoids are used as representative structures and become the default RMSD reference set. To also compute RMSD to known end states, set `user_specified_structures` in the settings.

In [ ]:
_banner("4. Clustering")

ref_structures = ens.cluster()

num_clusters = ens.df['cluster'].nunique() - (1 if -1 in ens.df['cluster'].values else 0)
n_noise      = (ens.df['cluster'] == -1).sum()

print(f"\n  Clusters : {num_clusters}  |  Noise points : {n_noise}")
print(f"  Medoids  : {ref_structures}")

cluster_best_pLDDT = [
    ens.df[ens.df['cluster'] == i].sort_values('pLDDT', ascending=False).index[0]
    for i in range(num_clusters)
]
print(f"\n  Top-pLDDT structure per cluster:")
#display(ens.df.loc[ens.df.index.isin(cluster_best_pLDDT)].sort_values('pLDDT', ascending=False))

if 'cluster' in ens.df.columns:
    g = sns.PairGrid(ens.df[pc_cols + ['cluster']], vars=pc_cols, hue='cluster',
                     height=1.5, palette='Paired')
    g.map(plt.scatter, marker='.', s=2)
    for rep in ref_structures:
        for i in range(len(pc_cols)):
            for j in range(len(pc_cols)):
                g.axes[i][j].scatter(
                    ens.df.loc[rep, pc_cols[j]], ens.df.loc[rep, pc_cols[i]],
                    c='black', marker='x', s=100, zorder=10,
                )
else:
    g = sns.PairGrid(ens.df[pc_cols], height=1.5)
    g.map(plt.scatter, marker='.', s=2)
plt.suptitle('PC space', y=1.02)
plt.show()

# ── PC space coloured by MSA depth ───────────────────────────────────────────
# Sampler filenames: r{round}_[cell_{i}_{j}_]{paired}-{unpaired}_rank_NNN.pdb
_msa_labels = ens.df.index.str.extract(r'_(\d+-\d+)_rank_', expand=False).fillna('other')
_msa_order  = sorted(_msa_labels.unique(),
                     key=lambda x: int(x.split('-')[0]) if x != 'other' else 0)

_plot_df = ens.df[pc_cols].copy()
_plot_df['msa'] = pd.Categorical(_msa_labels, categories=_msa_order, ordered=True)

g = sns.PairGrid(_plot_df, vars=pc_cols, hue='msa', hue_order=_msa_order,
                 height=1.5, palette='rainbow')
g.map(plt.scatter, marker='.', s=2, alpha=0.5)
g.add_legend(markerscale=4, fontsize=7, title='MSA depth')
plt.suptitle('PC space — MSA depth', y=1.02)
plt.show()

# bar plot - number of structures per MSA depth
_msa_counts = _msa_labels.value_counts().reindex(_msa_order, fill_value=0)
plt.figure(figsize=(6, 3))
sns.barplot(x=_msa_counts.index, y=_msa_counts.values, palette='rainbow')
plt.xticks(rotation=45)
plt.ylabel('Number of structures')
plt.title('Structures per MSA depth')
plt.show()

## 5. Structural Properties

Computes per-structure properties in parallel and adds them as columns to `ens.df`: RMSD to each reference structure, solvent-accessible surface area, and user-defined inter-atomic distances.

In [ ]:
_banner("5. Structural Properties")

_df_checkpoint = out_dir + 'ensemble_df.parquet'

_want_rmsd  = [f'rmsd_to_{r}' for r in ref_structures] if calculate_rmsd else []
_want_dists = list(distances_to_evaluate.keys())        if calculate_distances else []
_want_sasa  = ['sasa']                                  if calculate_sasa else []
_want_refs  = [f'rmsd_to_{k}' for k in user_specified_structures] if user_specified_structures else []
_want_all   = _want_rmsd + _want_dists + _want_sasa + _want_refs

# Restore any already-computed columns that are still valid for this ensemble
if os.path.exists(_df_checkpoint):
    _saved = pd.read_parquet(_df_checkpoint)
    if _saved.index.equals(ens.df.index):
        for c in [c for c in _want_all if c in _saved.columns and c not in ens.df.columns]:
            ens.df[c] = _saved[c]
    else:
        print(f"  Note: checkpoint has {len(_saved)} structures vs {len(ens.df)} — will recompute")

# Compute anything still missing (covers first run, changed threshold, changed ref structures)
_missing_rmsd  = [r for r in ref_structures if f'rmsd_to_{r}' not in ens.df.columns]
_missing_dists = {k: v for k, v in distances_to_evaluate.items()
                  if k not in ens.df.columns and all(v)} if calculate_distances else {}
_missing_sasa  = calculate_sasa and 'sasa' not in ens.df.columns
_missing_refs  = {k: v for k, v in user_specified_structures.items()
                  if f'rmsd_to_{k}' not in ens.df.columns} if user_specified_structures else {}

if _missing_rmsd or _missing_dists or _missing_sasa:
    ens.compute_metrics(
        ref_structures=_missing_rmsd,
        distances=_missing_dists or None,
        compute_sasa=_missing_sasa,
        num_processes=num_processes,
    )
    ens.df.to_parquet(_df_checkpoint)
    print(f"  Saved → {_df_checkpoint}")

if _missing_refs:
    ens.compute_reference_rmsds(_missing_refs, ref_resid_offset=user_ref_resid_offset,
                                num_processes=num_processes)
    ens.df.to_parquet(_df_checkpoint)
    print(f"  Saved → {_df_checkpoint}")

if not (_missing_rmsd or _missing_dists or _missing_sasa or _missing_refs):
    print(f"  Loaded from checkpoint")

print(f"\n  Ensemble dataframe  ({len(ens.df)} rows × {len(ens.df.columns)} cols):")
#display(ens.df)

# ── Closest structures to user-specified references ──────────────────────────
if user_specified_structures:
    print("\n  Closest ensemble structures to user-specified references:")
    for label in user_specified_structures:
        col = f'rmsd_to_{label}'
        closest = ens.df[col].idxmin()
        print(f"    {label}: {closest}  (RMSD = {ens.df.loc[closest, col]:.2f} Å)")
    print("\n  To use these as fixed end states in MC optimisation, set:")
    print("    fixed_endstates = True")
    print("  and in Section 8 (bin_structures), override the initial path endpoints manually.")

## 6. Visualisation

Plots ensemble density in PC space (hexbin), overlaid distance distributions, RMSD histograms, and SASA/pLDDT distributions. Reference structures are annotated with crosses.

In [ ]:
_banner("6. Visualisation")

valid_distances = []
if calculate_distances and distances_to_evaluate:
    valid_distances = [d for d in distances_to_evaluate if not ens.df[d].isna().all()]

n_hex_panels = 1 + len(valid_distances)
n_hist       = (1 if calculate_rmsd else 0) + (1 if calculate_sasa else 0) + 1
ncols        = n_hex_panels + n_hist

fig, axes = plt.subplots(1, ncols, figsize=(3 * ncols, 3))
axes = np.atleast_1d(axes)

# ── hexbin panels ──────────────────────────────────────────────────────────
hb = axes[0].hexbin(ens.df[plot_variable_1], ens.df[plot_variable_2],
                    cmap='plasma', gridsize=50, mincnt=1, bins='log')
axes[0].set_xlabel(plot_variable_1); axes[0].set_ylabel(plot_variable_2)
fig.colorbar(hb, ax=axes[0]); axes[0].set_title('Density')
for ref in ref_structures:
    axes[0].scatter(ens.df.loc[ref, plot_variable_1], ens.df.loc[ref, plot_variable_2],
                    c='black', marker='x', s=100)
    #axes[0].annotate(ref, (ens.df.loc[ref, plot_variable_1], ens.df.loc[ref, plot_variable_2]))

for i, distance_name in enumerate(valid_distances):
    hb = axes[1 + i].hexbin(ens.df[plot_variable_1], ens.df[plot_variable_2],
                             C=ens.df[distance_name], cmap='coolwarm', gridsize=50, mincnt=1)
    axes[1 + i].set_xlabel(plot_variable_1); axes[1 + i].set_ylabel(plot_variable_2)
    fig.colorbar(hb, ax=axes[1 + i], label=distance_name)
    axes[1 + i].set_title(distance_name)

# ── histogram panels ────────────────────────────────────────────────────────
ax_idx = n_hex_panels

if calculate_rmsd:
    for ref in ref_structures:
        axes[ax_idx].hist(ens.df[f'rmsd_to_{ref}'], bins=20, alpha=0.5, label=ref)
    #axes[ax_idx].legend(loc='upper right')
    axes[ax_idx].set_xlabel('RMSD'); axes[ax_idx].set_ylabel('Count')
    axes[ax_idx].set_xlim(0)
    ax_idx += 1

if calculate_sasa:
    axes[ax_idx].hist(ens.df['sasa'], bins=20, alpha=0.5, color='C3')
    axes[ax_idx].set_xlabel('SASA'); axes[ax_idx].set_ylabel('Count')
    ax_idx += 1

axes[ax_idx].hist(ens.df['pLDDT'], bins=20, alpha=0.5)
axes[ax_idx].set_xlabel('pLDDT')

plt.tight_layout()
plt.show()

In [ ]:
# ── Closest structures to user-specified references (PCA overlay) ─────────────
if user_specified_structures:
    _banner("6b. Closest Structures to User-Specified References")

    closest_structures = {}
    for label, ref_path in user_specified_structures.items():
        structure, rmsd = ens.get_closest_structure(
            ref_path, ref_resid_offset=user_ref_resid_offset, num_processes=num_processes
        )
        closest_structures[label] = (structure, rmsd)
        print(f"  {label}: {structure}  (RMSD = {rmsd:.2f} Å)")

    fig, ax = plt.subplots(figsize=(5, 4))
    hb = ax.hexbin(ens.df[plot_variable_1], ens.df[plot_variable_2],
                   cmap='plasma', gridsize=50, mincnt=1, bins='log')
    fig.colorbar(hb, ax=ax)

    colors_ref = plt.colormaps['tab10'](np.linspace(0, 0.4, len(closest_structures)))
    for (label, (structure, rmsd)), color in zip(closest_structures.items(), colors_ref):
        ax.scatter(ens.df.loc[structure, plot_variable_1],
                   ens.df.loc[structure, plot_variable_2],
                   color=color, marker='*', s=300, zorder=10, label=f'{label} ({rmsd:.2f} Å)')

    ax.set_xlabel(plot_variable_1)
    ax.set_ylabel(plot_variable_2)
    ax.set_title('Closest ensemble structures to references')
    ax.legend(loc='best')
    plt.tight_layout()
    plt.show()

## 7. Property Matrices

Builds the pairwise matrices used by the Monte Carlo path optimisation: the step-cost matrix
(Ramachandran φ/ψ circular distance, or RMSD), the CV difference matrix, and, when
`mc_wf_janin > 0`, the Janin χ1/χ2 sidechain distance matrix over the PCA selection.

In [ ]:
_banner("7. Property Matrices")

_matrices_checkpoint = out_dir + f'matrices_{step_matrix_type}.npz'
_n = len(ens.df)
_matrices_valid = False

if os.path.exists(_matrices_checkpoint):
    _npz = np.load(_matrices_checkpoint)
    _step_key = step_matrix_type + '_matrix'
    if (_step_key in _npz and _npz[_step_key].shape[0] == _n
            and str(_npz.get('cv', collective_variable)) == collective_variable
            and str(_npz.get('cv2', collective_variable_2)) == str(collective_variable_2)):
        setattr(ens, _step_key, _npz[_step_key])
        ens.cv_matrix  = _npz['cv_matrix']
        ens.cv2_matrix = _npz['cv2_matrix'] if 'cv2_matrix' in _npz else None
        _matrices_valid = True
        print(f"  Loaded from checkpoint")
    else:
        print(f"  Checkpoint size, type or CV mismatch — will recompute")

if not _matrices_valid:
    ens.compute_matrices(cv_column=collective_variable, cv2_column=collective_variable_2,
                         step_matrix=step_matrix_type)
    _step_key = step_matrix_type + '_matrix'
    _save = {_step_key: getattr(ens, _step_key), 'cv_matrix': ens.cv_matrix,
             'cv': collective_variable, 'cv2': str(collective_variable_2)}
    if ens.cv2_matrix is not None:
        _save['cv2_matrix'] = ens.cv2_matrix
    np.savez(_matrices_checkpoint, **_save)
    print(f"  Saved → {_matrices_checkpoint}")

# Optional Janin (sidechain χ1/χ2) matrix, over the same residues as the PCA selection
if mc_wf_janin > 0:
    from ensemblr.calc_matrices import get_janin_matrix
    _janin_checkpoint = out_dir + 'matrices_janin.npz'
    _npz = np.load(_janin_checkpoint) if os.path.exists(_janin_checkpoint) else None
    if (_npz is not None and _npz['janin_matrix'].shape[0] == _n
            and str(_npz['selection']) == ens.rmsd_selection):
        ens.janin_matrix = _npz['janin_matrix']
        print(f"  Janin matrix loaded from checkpoint")
    else:
        ens.janin_matrix = get_janin_matrix(ens.df, ens.structure_directory,
                                            selection=ens.rmsd_selection)
        np.savez(_janin_checkpoint, janin_matrix=ens.janin_matrix, selection=ens.rmsd_selection)
        print(f"  Saved → {_janin_checkpoint}")

_step_matrix = ens.rama_matrix if step_matrix_type == 'rama' else ens.rmsd_matrix
print(f"  Step matrix ({step_matrix_type}) : {_step_matrix.shape}  "
      f"min={_step_matrix.min():.2f}  max={_step_matrix.max():.2f}")

matrices = [ens.cv_matrix, _step_matrix]
titles   = ['CV matrix', f'{step_matrix_type.upper()} matrix']
if ens.cv2_matrix is not None:
    matrices.append(ens.cv2_matrix); titles.append('CV2 matrix')
if ens.janin_matrix is not None:
    matrices.append(ens.janin_matrix); titles.append('Janin matrix')

fig, axes = plt.subplots(1, len(matrices), figsize=(3 * len(matrices), 3))
axes = np.atleast_1d(axes)
for ax, matrix, title in zip(axes, matrices, titles):
    ax.imshow(matrix, cmap='inferno'); ax.set_title(title)
plt.tight_layout()
plt.show()

## 8. Path Optimisation

Bins structures along PC1 with KMeans, then runs `mc_n_runs` independent Monte Carlo simulated
annealing optimisations in parallel. The energy function penalises large backbone (Rama) and
sidechain (Janin) steps between adjacent windows and uneven spacing along PC1. Results are sorted by
total path energy.

In [ ]:
_banner("8. Path Optimisation — Binning")

finder = MCPathFinder(ens, n_bins=mc_n_bins, cv=collective_variable, cv2=collective_variable_2)
finder.bin_structures()

_bin_counts = ens.df['bin'].value_counts()
print(f"  Bins        : {mc_n_bins}")
print(f"  Structures/bin — min: {_bin_counts.min()}  max: {_bin_counts.max()}  "
      f"mean: {_bin_counts.mean():.1f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 3))

ax1.hist(ens.df['bin'], bins=mc_n_bins)
ax1.set_xlabel('Bin'); ax1.set_ylabel('Count'); ax1.set_title('Structures per bin')

hb = ax2.hexbin(ens.df[plot_variable_1], ens.df[plot_variable_2],
                C=ens.df['bin'], cmap='tab20c', reduce_C_function=np.amax)
fig.colorbar(hb, ax=ax2, label='bin')
for i, struct in enumerate(finder._initial_path):
    ax2.scatter(ens.df.loc[struct, plot_variable_1], ens.df.loc[struct, plot_variable_2],
                marker='x', color='black', s=10)
    ax2.annotate(i, (ens.df.loc[struct, plot_variable_1], ens.df.loc[struct, plot_variable_2]))
ax2.set_xlabel(plot_variable_1); ax2.set_ylabel(plot_variable_2); ax2.set_title('Bin assignment')

plt.tight_layout()
plt.show()

In [ ]:
_banner("8. Path Optimisation — Monte Carlo")

_mc_checkpoint = out_dir + 'mc_results.pkl'
_mc_weights_file = out_dir + 'mc_results_weights.json'
_mc_weights = {'step_matrix': step_matrix_type, 'wf_step': mc_wf_step, 'wf_cv': mc_wf_cv,
               'wf_odf': mc_wf_odf if mc_wf_odf is not None else 0.0, 'wf_janin': mc_wf_janin}

def _check_weights(path, current):
    """Report weights saved alongside a checkpoint and flag any that differ from the settings."""
    if not os.path.exists(path):
        print("  (no saved weights for this checkpoint)")
        return
    with open(path) as f:
        saved = json.load(f)
    print(f"  Saved weights : {saved}")
    changed = {k: (saved.get(k), v) for k, v in current.items() if saved.get(k) != v}
    if changed:
        print(f"  WARNING: settings differ from checkpoint {changed} — delete the checkpoint to rerun")

if os.path.exists(_mc_checkpoint):
    with open(_mc_checkpoint, 'rb') as f:
        _mc_saved = pickle.load(f)
    _mc_structs = {s for row in _mc_saved.itertuples() for s in row.path}
    _ens_structs = set(ens.df.index)
    if _mc_structs.issubset(_ens_structs) and len(_mc_saved.iloc[0]["path"]) == mc_n_bins:
        finder.runs_df = _mc_saved
        print(f"  Loaded from checkpoint : {_mc_checkpoint}")
        _check_weights(_mc_weights_file, _mc_weights)
    else:
        n_missing = len(_mc_structs - _ens_structs)
        print(f"  Note: MC checkpoint has {n_missing} unknown structures or wrong bin count — will rerun")
else:
    print(f'  wf_step={mc_wf_step}  wf_cv={mc_wf_cv}  wf_odf={mc_wf_odf if mc_wf_odf is not None else 0.0:.3f}  wf_janin={mc_wf_janin}')
    _wf_odf = mc_wf_odf if mc_wf_odf is not None else 0.0
    finder.run(
        n_runs=mc_n_runs,
        num_processes=num_processes,
        reproducible_seeds=reproducible_seeds,
        fixed_endpoints=fixed_endstates,
        wf_step=mc_wf_step,
        wf_cv=mc_wf_cv,
        wf_odf=_wf_odf,
        wf_janin=mc_wf_janin,
    )
    with open(_mc_checkpoint, 'wb') as f:
        pickle.dump(finder.runs_df, f)
    with open(_mc_weights_file, 'w') as f:
        json.dump(_mc_weights, f, indent=1)
    print(f"  Saved checkpoint → {_mc_checkpoint}")

print(f"\n  Best path energy  : {finder.runs_df.iloc[ 0]['energy']:.4f}")
print(f"  Worst path energy : {finder.runs_df.iloc[-1]['energy']:.4f}")

_row = {s: i for i, s in enumerate(ens.df.index)}

def _path_steps(path, matrix):
    return np.array([matrix[_row[a], _row[b]] for a, b in zip(path[:-1], path[1:])])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3))

_best, _worst = finder.runs_df.iloc[0]['path'], finder.runs_df.iloc[-1]['path']
ax1.plot(_path_steps(_best,  _step_matrix), color='black', label=f'{step_matrix_type} (best)')
ax1.plot(_path_steps(_worst, _step_matrix), color='black', alpha=0.3, label=f'{step_matrix_type} (worst)')
if ens.janin_matrix is not None:
    ax1.plot(_path_steps(_best,  ens.janin_matrix), color='C4', label='Janin (best)')
    ax1.plot(_path_steps(_worst, ens.janin_matrix), color='C4', alpha=0.3, label='Janin (worst)')
ax1.set_xlabel('Window'); ax1.set_ylabel('Step distance'); ax1.legend(fontsize=7)
ax1.set_title('Best vs worst path')

for i in range(1, mc_n_runs):
    ax2.plot(finder.runs_df.iloc[i]['relaxation_energies'], alpha=0.3, color='steelblue', lw=0.8)
ax2.plot(finder.runs_df.iloc[0]['relaxation_energies'], color='black', lw=1.5, label='best')
ax2.set_xlabel('MC step'); ax2.set_ylabel('Path energy')
ax2.set_title('MC relaxation curves'); ax2.legend()

plt.tight_layout()
plt.show()

finder.write_paths(out_dir + 'mc_paths/', n=mc_n_paths_to_save, resid_offset=resid_offset)

## 9. Path Assessment

Visualises the best MC path in PC space and compares actual PC1 values at each window against the
ideal linearly-spaced values. Large offsets suggest the MC struggled in that region (often a gap in
the ensemble) — consider increasing `mc_n_runs`, lowering `mc_n_bins`, or filling the gap with
`ensemblr-generate --fill`.

In [ ]:
_banner("9. Path Assessment")

actual_window_values, ideal_window_values = finder.get_window_values()
offsets = np.abs(ideal_window_values - actual_window_values)

print(f"  Mean absolute offset from ideal : {offsets.mean():.4f}")
print(f"  Max  absolute offset from ideal : {offsets.max():.4f}  (window {offsets.argmax()})")

fig, axs = plt.subplots(1, 3, figsize=(11, 3))

# panel 1: best path in PC space
hb = axs[0].hexbin(ens.df[plot_variable_1], ens.df[plot_variable_2],
                   C=ens.df['bin'], cmap='tab20c', reduce_C_function=np.amax)
best_path = finder.runs_df.iloc[0]['path']
for struct in best_path:
    axs[0].scatter(ens.df.loc[struct, plot_variable_1], ens.df.loc[struct, plot_variable_2],
                   c='black', s=32)
for i in range(len(best_path) - 1):
    x1, y1 = ens.df.loc[best_path[i],   plot_variable_1], ens.df.loc[best_path[i],   plot_variable_2]
    x2, y2 = ens.df.loc[best_path[i+1], plot_variable_1], ens.df.loc[best_path[i+1], plot_variable_2]
    axs[0].plot([x1, x2], [y1, y2], c='black', alpha=0.6, linestyle='dashed')
axs[0].set_xlabel(plot_variable_1); axs[0].set_ylabel(plot_variable_2)
axs[0].set_title('Best path in PC space')

# panel 2: ideal vs actual CV per window
axs[1].plot(ideal_window_values,  label='ideal',  color='black', alpha=0.2)
axs[1].scatter(range(mc_n_bins), ideal_window_values, c=range(mc_n_bins), cmap='coolwarm', alpha=0.5)
axs[1].plot(actual_window_values, label='actual', color='black', alpha=0.8)
axs[1].scatter(range(mc_n_bins), actual_window_values, c=range(mc_n_bins), cmap='coolwarm')
for i in range(mc_n_bins):
    axs[1].plot([i, i], [ideal_window_values[i], actual_window_values[i]],
                c='black', alpha=0.2, linestyle='dashed')
axs[1].set_xlabel('Window'); axs[1].set_ylabel(collective_variable)
axs[1].legend(); axs[1].set_title('Ideal vs actual CV')

# panel 3: per-window offset
axs[2].bar(range(mc_n_bins), offsets, color='black', alpha=0.5)
axs[2].axhline(offsets.mean(), color='red', linestyle='dashed', label=f'mean={offsets.mean():.3f}')
axs[2].set_xlabel('Window'); axs[2].set_ylabel('|offset|')
axs[2].set_title('Offset from ideal'); axs[2].legend()

plt.tight_layout()
plt.show()

## 10. Umbrella Sampling Setup

Embeds each window structure into the simulation template (protein + lipids + water) by replacing the protein coordinates, then writes PLUMED input files for both multi-directory and single-window runs.

**Note:** Always inspect output windows for clashes, particularly threaded lipids. The template should come from a prior simulation with a working topology.

In [ ]:
_banner("10. Umbrella Sampling Setup")

if not template_pdb_for_umbrella_sampling:
    print("  No template_pdb_for_umbrella_sampling set — skipping. "
          "The best path is in mc_paths/; set a template and re-run this cell.")
else:
    prep = UmbrellaSamplingPrep(
        path_finder=finder,
        ensemble=ens,
        template_pdb=template_pdb_for_umbrella_sampling,
        output_dir=out_dir + 'umbrella_sampling/',
    )

    prep.prepare_windows(resid_offset=resid_offset)
    print(f"\n  Windows written → {out_dir}umbrella_sampling/")

    prep.write_plumed_input(
        distances_dict=distances_to_evaluate if calculate_distances else None,
        resid_offset=resid_offset,
    )
    print(f"  PLUMED input written")

## 11. Debugging (Optional)

Compares CV values reported by PLUMED (from `COLVAR` files written during a short test run) against the MDA-computed projections. Run after an initial short simulation to verify PLUMED tracks the expected CV before committing to a full production run.

In [ ]:
_banner("11. Debugging — PLUMED CV verification")

import MDAnalysis as mda

umbrella_sampling_directory = out_dir + 'umbrella_sampling/'
colvar_files = glob.glob(umbrella_sampling_directory + 'window_*/COLVAR')
colvar_files.sort(key=lambda x: int(x.split('/')[-2].split('_')[1]))

actual_window_values, cv_values_for_plumed_input = finder.get_window_values()

n_sel = len(mda.Universe(out_dir + 'ensemble.pdb').select_atoms(ens.rmsd_selection))
print(f"  Atoms in rmsd_selection : {n_sel}")
print(f"  COLVAR files found      : {len(colvar_files)}")

if not colvar_files:
    print("\n  No COLVAR files found — run a short test simulation first.")
else:
    cv_values = []
    for file in colvar_files:
        with open(file) as f:
            lines = f.readlines()
            cv_values.append(float(lines[1].split()[1]))

    _max_dev = max(abs(cv_values[i] - actual_window_values[i]) for i in range(len(cv_values)))
    print(f"  Max |PLUMED − MDA| deviation : {_max_dev:.4f}")

    plt.figure(figsize=(6, 3))
    plt.plot(actual_window_values,       label='MDA actual', alpha=0.7, color='red')
    plt.plot(cv_values_for_plumed_input, label='ideal',      alpha=0.5, color='black', linestyle='dashed')
    plt.plot(cv_values,                  label='PLUMED',     alpha=0.7, color='steelblue')
    plt.xlabel('Window'); plt.ylabel(collective_variable)
    plt.title('CV cross-check: MDA vs PLUMED'); plt.legend()
    plt.tight_layout()
    plt.show()